# ND-AWS under amplitude damping, n = 20

Sweeps the amplitude-damping strength `q` on a 20-qubit noisy simulator and compares the
noise-directed and standard warm-start ansatzes at each `q`. One Hamiltonian class per run;
`q = 0.0` is the noiseless member of the same family, produced by the same code path, so the
whole sweep shares one gate set and one compilation.

**Writes**: one dataset per (`q`, ansatz) pair into the standardized result store, each
holding one experiment set per (instance, restart). The store is
`DEFAULT_STORAGE_DIRECTORY`; with that variable unset or blank it is `output/` at the
repository root, no matter which folder the notebook was started from.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from datetime import datetime

import pandas as pd
import qiskit

import quapopt
from quapopt import ancillary_functions as anf
from quapopt.ancillary_functions.presets import (create_uncorrelated_noise_model_amplitude_damping,
                                                 generate_random_hamiltonian,
                                                 get_hamiltonian_class_description,
                                                 get_standard_subfolders_hierarchy_full,
                                                 ClassicalHamiltonian)
from quapopt.ancillary_functions.presets.ndar import (build_local_sampler_callable_and_updater_NDAR,
                                                      build_universal_logging_callable,
                                                      LocalSamplerName)
from quapopt.ancillary_functions.presets.wrapped_optimization import get_classical_optimizer_p1_default
from quapopt.circuits import backend_utilities as bck_utils
from quapopt.circuits.backend_utilities.qiskit.qiskit_config import DEFAULT_SIMULATOR_BACKEND_KWARGS
from quapopt.circuits.gates.native.NativeGateBuilderHeron import NativeGateBuilderHeron
from quapopt.data_analysis.data_handling import (HamiltonianModels,
                                                 ResultsLogger,
                                                 STANDARD_NAMES_DATA_TYPES as SNDT,
                                                 STANDARD_NAMES_VARIABLES as SNV)
from quapopt.data_analysis.data_handling.io_utilities.dataset_management import read_dataset_info
from quapopt.meta_algorithms.NDAR import ConvergenceCriterion, ConvergenceCriterionNames
from quapopt.meta_algorithms.NDAR.NDARRunner import NDARRunner
from quapopt.optimization.QAOA import InitialStateType, MixerType, PhaseSeparatorType, QubitMappingType

## Configuration

The only cell meant to be edited.

`Q_VALUES` is the damping sweep; `0.0` is the noiseless reference member of the family.
`ANSATZES` lists the arms to run at each `q`. The sweep runs the full product
`Q_VALUES x ANSATZES`, one dataset each.

`NOISE_MODEL` selects which channel is attached:

- `'paper'` — amplitude damping after every single-qubit gate and no two-qubit error; the
  channel behind the paper's damping sweep.
- `'full'` — damping after every gate.

`RUN_ID` is the suffix of every dataset name the sweep writes. There is no auto-suffix: a
`RUN_ID` that already holds data stops the notebook before anything runs.

This sweep runs with `add_hdls = False`, without the greedy step the 100-qubit study applies
between iterations. That step is a Hamming-distance local search over the neighbourhood of
each sample; the paper calls its distance-2 case Hamming Distance Quadratic Search (HDQS),
which is what `add_hdls` switches on. Leaving it off keeps these curves a measurement of the
ansatz alone.

In [ ]:
MODEL = 'RG3'
# 'ER10' | 'ER20' | 'RG3'

Q_VALUES = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1]

ANSATZES = ['ND', 'Standard']

NOISE_MODEL = 'paper'
# 'paper' | 'full'

RUN_ID = 'run-01'

hamiltonian_indices_range = list(range(10))
restarts_range = range(3)
number_of_samples_qiskit = 100

## Problem, backend and gate set

| MODEL | class | instance kwargs |
|---|---|---|
| ER10 | Erdos-Renyi, p = 0.1 | `{'p_or_M': 0.1}` |
| ER20 | Erdos-Renyi, p = 0.2 | `{'p_or_M': 0.2}` |
| RG3 | 3-regular | `{'graph_degree': 3}` |

Twenty qubits, one p=1 phase-separator layer implementing the whole Hamiltonian
(`time_block_size = 1.0`).

Backend: a bare `AerSimulator` at every `q`, including `q = 0`. The sampler primitive does
not transpile, so the circuit must already be in the basis the noise model is defined on —
`x, sx, rz, id, cz` — for "damping after each gate" to hold. The notebook therefore compiles
with the Heron native gate builder and SABRE routing at every `q`, so that one gate set is
used across the whole sweep. A gate outside the noise model's basis would run noiselessly.

In [ ]:
MODEL_TABLE = {
    'ER10': {'hamiltonian_model': HamiltonianModels.ErdosRenyi, 'instance_kwargs': {'p_or_M': 0.1}},
    'ER20': {'hamiltonian_model': HamiltonianModels.ErdosRenyi, 'instance_kwargs': {'p_or_M': 0.2}},
    'RG3': {'hamiltonian_model': HamiltonianModels.RegularGraph, 'instance_kwargs': {'graph_degree': 3}},
}

if MODEL not in MODEL_TABLE:
    raise ValueError(f"unknown MODEL {MODEL!r}; choose one of {sorted(MODEL_TABLE)}")
if NOISE_MODEL not in ('paper', 'full'):
    raise ValueError(f"unknown NOISE_MODEL {NOISE_MODEL!r}; choose 'paper' or 'full'")
for _a in ANSATZES:
    if _a not in ('ND', 'Standard'):
        raise ValueError(f"unknown ansatz {_a!r} in ANSATZES; choose from 'ND', 'Standard'")

hamiltonian_model = MODEL_TABLE[MODEL]['hamiltonian_model']
instance_kwargs = MODEL_TABLE[MODEL]['instance_kwargs']

number_of_qubits = 20
localities = (2,)
time_block_size = 1.0
backend_name = 'aer'
simulation = True
mps_simulation = False
max_bond_dimension = None
add_hdls = False
two_qubit_damping = (NOISE_MODEL == 'full')

gate_builder = NativeGateBuilderHeron()
qubit_mapping_type = QubitMappingType.sabre
pass_manager_seeds_list = [42]

qiskit_backend = bck_utils.get_qiskit_backend(backend_name=backend_name)

# Twenty qubits: the instances are built on the CPU backend, so this study needs no GPU.
default_backend = 'numpy'

# The class description names the folder the datasets are written under; it depends on the
# model and the localities alone, so it is taken from the generator rather than from an
# instance.
ham_class_description = get_hamiltonian_class_description(hamiltonian_model=hamiltonian_model,
                                                          localities=localities)

print(default_backend, ham_class_description)

## Ansatz, angle optimizer and the transpiler seed

Angles are set offline: for each NDAR iteration the sampler optimizes gamma on a
reduced-density-matrix simulator with COBYQA plus basinhopping (200 function calls), and
finds the best beta for each gamma internally.

The published runs found that beta by a scan over a grid of `betas_search_space_size`
values. `analytical_betas = False` keeps that scan, so this notebook runs the published
configuration. The library can now also find that beta exactly, in closed form, when the
warm-start bias is the same on every qubit, as it is here; `analytical_betas = None`
selects it.

Basinhopping takes random displacement steps, so the search has a seed of its own,
`optimizer_seed`. `None` keeps the seed the optimizer preset configures, which is what makes
a re-run return the same angles; an integer overrides it.

NDAR stops when the best cost has not improved for three consecutive iterations, or when
the known optimum is sampled. At 20 qubits the optimum is known, so the run also breaks as
soon as a ground state is drawn.

Transpiler seed: kept equal to the 100-qubit runs. It has no effect on a bare
`AerSimulator`.

In [ ]:
phase_separator_type = PhaseSeparatorType.QAOA
mixer_type = MixerType.ws_qaoa_identical

# Fallback only: the kwargs updater supplies a seed on every NDAR iteration, so this value
# is what a caller that passes no schedule would get. See time_block_seed_callable below.
time_block_seed = -1
efficient_finding_of_best_beta = True
betas_search_space_size = 10 ** 6
# False: find beta by the grid scan over betas_search_space_size values, as the published
# runs did. None: the closed-form beta, where it applies.
analytical_betas = False
number_of_function_calls_simulator = 200
r_best_results = 1
break_after_finding_ground_state = True

# The seed of the offline angle search itself. None keeps the seed the optimizer preset
# configures, which is what makes the search reproducible; an integer overrides it. The
# effective value is in optimizer_metadata['basinhopping_kwargs'] either way.
optimizer_seed = None

convergence_value = 3
convergence_criterion = ConvergenceCriterion(convergence_criterion_name=ConvergenceCriterionNames.MaxUnsuccessfulTrials,
                                             convergence_value=convergence_value)

classical_optimizer, optimizer_metadata = get_classical_optimizer_p1_default(
    efficient_finding_of_best_beta=efficient_finding_of_best_beta,
    number_of_qubits=number_of_qubits)

optimizer_metadata['betas_search_space_size'] = betas_search_space_size
optimizer_metadata['analytical_betas'] = analytical_betas
optimizer_metadata['optimizer_seed'] = optimizer_seed

pass_manager_kwargs_main = {'optimization_level': 3 if qubit_mapping_type == QubitMappingType.sabre else 0}

simulator_config_dict = {'classical_optimizer': classical_optimizer,
                         'betas_search_space_size': betas_search_space_size,
                         'analytical_betas': analytical_betas,
                         'efficient_finding_of_best_beta': efficient_finding_of_best_beta,
                         'optimizer_seed': optimizer_seed,
                         }

## Bias schedule and phase-separator subset

The 20-qubit study uses a single bias value per iteration: `0.5` at iteration 0, `0.1` for
iterations 1 to 3, `0.05` from iteration 4 on.

`time_block_seed_callable` selects which interactions enter the phase-separator layer:
iterations 0 to 3 take the largest-magnitude subset (`-1`), and every later iteration takes
a fresh random subset drawn with the iteration index as seed. This sweep never uses it: at
`time_block_size = 1.0` the layer implements the whole Hamiltonian in its stored order and
the seed does not reach the circuit at all. The metadata row records that, rather than a
seed value that had no effect.

In [ ]:
def bias_list_iteration_callable(ndar_iteration_index):
    if ndar_iteration_index == 0:
        return [0.5]
    elif ndar_iteration_index <= 3:
        return [0.1]
    else:
        return [0.05]


# From this iteration on, the phase-separator subset is drawn at random with the iteration
# index as seed; before it, -1 takes the largest-magnitude subset.
time_block_seed_random_from_iteration = 4


def time_block_seed_callable(ndar_iteration_index):
    if ndar_iteration_index < time_block_seed_random_from_iteration:
        return -1
    else:
        return ndar_iteration_index


# What the metadata row records, built from the same constant the callable uses so the two
# cannot drift apart.
if time_block_size == 1.0:
    time_block_seed_schedule = ('unused: at time_block_size=1.0 the phase separator holds '
                                'every interaction and the seed does not reach the circuit')
else:
    time_block_seed_schedule = (f'-1 for iterations 0 to {time_block_seed_random_from_iteration - 1}, '
                                f'then the NDAR iteration index')

## Dataset names and the name guard

`q = 0` writes `NDAWS_simulation_noiseless_<class>[_no-gauges-version]_<RUN_ID>`; every
`q > 0` writes `NDAWS_simulation_noisy_<class>_NM=amplitude-damping_AMP=<q>[_no-gauges-version]_<RUN_ID>`.

The guard refuses a name that already holds experiment sets — mixing two runs under one name
would make the analyzer's best-of-all-sets selection silently span both. The whole sweep is
checked before the first run, so a clash costs no compute.

In [ ]:
def build_dataset_name(q, ansatz):
    noiseless = (q == 0)
    no_gauges = (ansatz == 'Standard')

    name = "NDAWS_simulation_"
    name += 'noiseless_' if noiseless else 'noisy_'

    if time_block_size != 1.0:
        name += f'k={time_block_size}_'

    if hamiltonian_model == HamiltonianModels.ErdosRenyi:
        name += f"ER-{instance_kwargs['p_or_M']}_"
    elif hamiltonian_model == HamiltonianModels.RegularGraph:
        name += f"RG-{instance_kwargs['graph_degree']}_"
    else:
        raise NotImplementedError(f"Hamiltonian model {hamiltonian_model} is not implemented.")

    if not noiseless:
        name += f'NM=amplitude-damping_AMP={q}_'

    if no_gauges:
        name += 'no-gauges-version_'

    if add_hdls:
        name += 'HDLS_'

    return name + RUN_ID


def guard_dataset_name(name):
    existing_ids = read_dataset_info(dataset_name=name)
    if len(existing_ids) > 0:
        raise ValueError(f"dataset {name} already holds {len(existing_ids)} "
                         f"experiment sets; choose a new RUN_ID")


SWEEP = [(q, ansatz) for q in Q_VALUES for ansatz in ANSATZES]

print('DATASET NAMES:')
for q, ansatz in SWEEP:
    _name = build_dataset_name(q, ansatz)
    guard_dataset_name(_name)
    print(f'  q={q}  {ansatz:<8}  {_name}')

## Run the sweep

For each (`q`, ansatz) pair: build the noise model, attach it to the sampler options, and run
ten instances times three restarts. `seed_simulator` is `1000 * instance + restart`, so the
shot stream is reproducible and is the same across the sweep for a given (instance, restart).

In [ ]:
all_experiment_set_ids = {}

for q, ansatz in SWEEP:
    noiseless_simulation = (q == 0)
    initial_state_type = (InitialStateType.ws_qaoa_identical if ansatz == 'ND'
                          else InitialStateType.ws_qaoa_general)
    no_gauges_version = initial_state_type == InitialStateType.ws_qaoa_general

    dataset_name = build_dataset_name(q, ansatz)
    guard_dataset_name(dataset_name)

    print("\n\n\n")
    anf.cool_print("q parameter", q, 'yellow')
    anf.cool_print("dataset", dataset_name, 'yellow')
    print("\n\n\n")

    if noiseless_simulation:
        noise_model_name = None
        amplitude_damping_probability = None
        noise_model = None
    else:
        noise_model_name = 'amplitude-damping'
        amplitude_damping_probability = q
        noise_model = create_uncorrelated_noise_model_amplitude_damping(
            number_of_qubits=number_of_qubits,
            amplitude_damping_probability=amplitude_damping_probability,
            two_qubit_damping=two_qubit_damping)

    experiments_folders_hierarchy = get_standard_subfolders_hierarchy_full(experiment_category='NDAR',
                                                                           experiment_subcategory='WS-QAOA-p1',
                                                                           backend_name=backend_name,
                                                                           simulation=simulation,
                                                                           noiseless_simulation=noiseless_simulation,
                                                                           hamiltonian_class_description=ham_class_description)

    if qiskit_backend.name[0:3] == 'aer':
        qiskit_sampler_options = DEFAULT_SIMULATOR_BACKEND_KWARGS.copy()
    else:
        raise ValueError(f"Backend {qiskit_backend.name} not recognized for simulation. ")

    if noise_model is not None:
        qiskit_sampler_options['noise_model'] = noise_model

    ansatz_config_dict = {'time_block_size': time_block_size,
                          'time_block_seed': time_block_seed if isinstance(time_block_seed, int) else None,
                          'phase_separator_type': phase_separator_type,
                          'mixer_type': mixer_type,
                          'initial_state_type': initial_state_type,
                          }

    qiskit_config_dict_main = {'qubit_mapping_type': qubit_mapping_type,
                               'qiskit_backend': qiskit_backend,
                               'backend_name': backend_name,
                               'pass_manager_seeds_list': pass_manager_seeds_list,
                               'gate_builder': gate_builder,
                               'qubit_indices_physical': None,
                               'simulation': simulation,
                               'noiseless_simulation': noiseless_simulation,
                               'mps_simulation': mps_simulation,
                               'max_bond_dimension': max_bond_dimension,
                               'no_gauges_version': no_gauges_version,
                               'qiskit_sampler_options': qiskit_sampler_options,
                               'add_hdls': add_hdls,
                               }

    qiskit_metadata_additional_main = {'qiskit_version': qiskit.__version__,
                                       'backend_name': backend_name,
                                       'simulation': simulation,
                                       'noiseless_simulation': noiseless_simulation,
                                       'gate_builder_class': gate_builder.__class__.__name__,
                                       'pass_manager_seeds_list': pass_manager_seeds_list,
                                       'noise_model_name': noise_model_name,
                                       'noise_model': noise_model.to_dict(
                                           serializable=True) if noise_model is not None else None,
                                       'amplitude_damping_probability': amplitude_damping_probability,
                                       }

    config_qiskit_sampler_main = {'r_best_results': r_best_results,
                                  'number_of_function_calls_simulator': number_of_function_calls_simulator,
                                  'number_of_samples_qiskit': number_of_samples_qiskit,
                                  'optimize_bias_offline': False,
                                  'bias_list_iteration_callable': bias_list_iteration_callable,
                                  'time_block_seed_callable': time_block_seed_callable,
                                  'time_block_seed_schedule': time_block_seed_schedule,
                                  'ansatz_config': ansatz_config_dict,
                                  'simulator_config': simulator_config_dict,
                                  }

    ndar_metadata = {'local_samplers': [LocalSamplerName.QiskitWithOfflineAnglesP1.value],
                     'convergence_criterion': convergence_criterion.get_description_string(),
                     }

    all_experiment_set_ids[dataset_name] = {i: [] for i in hamiltonian_indices_range}

    for seed_restart in restarts_range:
        anf.wait_unless_interrupted(5)
        for hamiltonian_instance_index in hamiltonian_indices_range:
            anf.wait_unless_interrupted(1)

            cm_filtered = None
            pass_manager_kwargs = pass_manager_kwargs_main.copy()
            pass_manager_kwargs['coupling_map'] = cm_filtered

            qiskit_config_dict = qiskit_config_dict_main.copy()
            qiskit_config_dict['pass_manager_kwargs'] = pass_manager_kwargs

            qiskit_metadata_additional = qiskit_metadata_additional_main.copy()
            qiskit_metadata_additional['pass_manager_kwargs'] = pass_manager_kwargs

            config_qiskit_sampler = config_qiskit_sampler_main.copy()

            qiskit_config_dict['qiskit_sampler_options'][
                'seed_simulator'] = 1000 * hamiltonian_instance_index + seed_restart

            config_qiskit_sampler['qiskit_config'] = qiskit_config_dict

            cost_hamiltonian_i: ClassicalHamiltonian = generate_random_hamiltonian(hamiltonian_model=hamiltonian_model,
                                                                                   number_of_qubits=number_of_qubits,
                                                                                   seed=hamiltonian_instance_index,
                                                                                   default_backend=default_backend,
                                                                                   localities=localities,
                                                                                   instance_kwargs=instance_kwargs)
            #The analysis notebooks read both extremal energies of the instance from the store, and the
            #ground-state break needs the lowest one; a re-run reads them instead of solving again.
            if cost_hamiltonian_i.lowest_energy is None or cost_hamiltonian_i.highest_energy is None:
                cost_hamiltonian_i.solve_hamiltonian(both_directions=True)
            cost_hamiltonian_i.write_to_file()

            ham_clas_description = cost_hamiltonian_i.hamiltonian_class_description
            ham_instance_description = cost_hamiltonian_i.hamiltonian_instance_description

            hamiltonian_instance_metadata = cost_hamiltonian_i.hamiltonian_metadata
            cost_hamiltonian_i._spectrum = None

            anf.cool_print('RUNNING NEW HAMILTONIAN!', '...', 'red')

            experiment_set_id = f"{anf.create_random_uuid()}"
            all_experiment_set_ids[dataset_name][hamiltonian_instance_index].append(experiment_set_id)

            universal_logger_callable = build_universal_logging_callable()

            logger_kwargs_main = {'experiment_folders_hierarchy': experiments_folders_hierarchy,
                                  'experiment_set_id': experiment_set_id,  #Used to group the experiments
                                  'dataset_name': dataset_name,
                                  }

            experiment_metadata = {"timestamp_start": datetime.now().isoformat(),
                                   "quapopt_version": getattr(quapopt, '__version__', 'unknown'),
                                   "python_version": sys.version.split()[0], }

            metadata_experiment_set = {"LoggerMetadata": logger_kwargs_main,
                                       "NDARMetadata": ndar_metadata,
                                       'ClassicalOptimizerMetadataQAOA': optimizer_metadata,
                                       'HamiltonianMetadata': hamiltonian_instance_metadata,
                                       'ExperimentMetadata': experiment_metadata,
                                       'ImplementationMetadataAdditional': qiskit_metadata_additional_main,
                                       'ImplementationMetadata': config_qiskit_sampler,
                                       "NoGaugesVersion": no_gauges_version,
                                       }

            with bck_utils.create_qiskit_session(qiskit_backend=qiskit_backend,
                                                 mocked=simulation,
                                                 mode='batch') as session_ibm:

                config_qiskit_sampler['qiskit_config']['session_ibm'] = session_ibm

                sampler_qiskit, updater_qiskit = build_local_sampler_callable_and_updater_NDAR(
                    local_sampler_name=LocalSamplerName.QiskitWithOfflineAnglesP1,
                    **config_qiskit_sampler,
                    verbosity=0,
                    logger_kwargs_main=logger_kwargs_main)

                metadata_experiment_set[
                    f"{LocalSamplerName.QiskitWithOfflineAnglesP1.value}Metadata"] = config_qiskit_sampler
                metadata_experiment_set[
                    f"{LocalSamplerName.QiskitWithOfflineAnglesP1.value}MetadataAdditional"] = qiskit_metadata_additional

                local_sampler_functions = [(sampler_qiskit,
                                            updater_qiskit,
                                            universal_logger_callable,
                                            LocalSamplerName.QiskitWithOfflineAnglesP1.value)]

                metadata_logger = ResultsLogger(**logger_kwargs_main,
                                                experiment_instance_id='nan')
                try:
                    metadata_logger.write_metadata(metadata=metadata_experiment_set,
                                                   data_type=SNDT.CircuitsMetadata,
                                                   shared_across_experiment_set=True)
                except FileExistsError:
                    anf.cool_print("WARNING:",
                                   "The experiment set already exists. The results will be appended to the existing files.",
                                   'red')

                df_metadata_simple_i = pd.DataFrame(data={SNV.ExperimentSetID.id_long: [experiment_set_id],
                                                          SNV.NumberOfQubits.id_long: [number_of_qubits],
                                                          SNV.HamiltonianClassDescription.id_long: [
                                                              ham_clas_description],
                                                          SNV.HamiltonianInstanceDescription.id_long: [
                                                              ham_instance_description],
                                                          'rBestSolutions': [r_best_results],
                                                          'TranspilerSeeds': [pass_manager_seeds_list],
                                                          SNV.TimeBlockSize.id_long: [time_block_size],
                                                          f"{SNV.TimeBlock.id_long}{SNV.Seed.id_long}": [
                                                              time_block_seed_schedule],
                                                          'NumberOfSamples': [number_of_samples_qiskit],
                                                          'LocalSamplers': [
                                                              LocalSamplerName.QiskitWithOfflineAnglesP1.value],
                                                          'CouplingMapIsFiltered': [cm_filtered is not None],
                                                          })

                metadata_logger.write_results(dataframe=df_metadata_simple_i,
                                              data_type=SNDT.QAOAOptimizationMetadata)

                ndar_runner = NDARRunner(input_hamiltonian_representation=cost_hamiltonian_i,
                                         convergence_criterion=convergence_criterion)

                anf.cool_print("EXPERIMENT SET ID:", experiment_set_id, 'yellow')

                best_res_ndar, history_ndar = ndar_runner.run_NDAR(
                    local_sampler_functions=local_sampler_functions,
                    show_progress_bar_ndar=True,
                    verbosity=1,
                    optimize_over_r_gauges=r_best_results,
                    break_after_finding_ground_state=break_after_finding_ground_state)

print('done:', sorted(all_experiment_set_ids))